# 01 — Dữ liệu, Khám phá EDA và Tiền xử lý

Mục tiêu của notebook này gồm 4 giai đoạn chuẩn mực:
1. **Xác định bài toán & Khám phá tổng quan (Data Understanding):** Đọc dữ liệu thô `housing.csv`, nhận diện các trường thông tin, kiểm tra kích thước và tỷ lệ khuyết thiếu ban đầu.
2. **Phân chia tập độc lập (Early Split 8:1:1):** Chia tách Train (80%), Validation (10%) và Test (10%) bằng phân tầng (Stratified) dựa trên phân vị giá nhà.
3. **Khám phá chuyên sâu chỉ trên tập Train (In-Depth EDA):** Trực quan hóa phân phối đặc trưng, bản đồ địa lý California, ma trận tương quan nhiệt Pearson và đặc trưng phân loại vị trí biển.
4. **Khớp tiền xử lý & Bàn giao artifacts:** Fit `ColumnTransformer` chỉ trên tập Train và lưu trữ các tệp thô cùng bộ biến đổi vào `notebooks/artifacts/`.

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'src' / 'config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_DIR = ROOT / 'notebooks' / 'artifacts'
TARGET = 'median_house_value'
SEED = 42

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_files(manifest):
    for name, expected in manifest['files'].items():
        if file_hash(ARTIFACT_DIR / name) != expected:
            raise ValueError(f'Artifact mismatch: {name}. Rerun notebooks in order.')

from src.data.loader import load_housing_data, fetch_dataset_info
from src.data.preprocessing import split_data_8_1_1, fit_preprocessor, transform_features, save_processed_splits
from src.models.persistence import save_preprocessor, load_preprocessor
from src.evaluation.visualization import (
    plot_feature_distributions, plot_geographical_distribution,
    plot_correlation_heatmap, plot_target_distribution, plot_categorical_distribution,
)
%matplotlib inline


## 1. Xác định bài toán và khám phá tổng quan tập dữ liệu thô (Data Understanding)

Dữ liệu được nạp từ tệp `data/raw/housing.csv`. Hàm `load_housing_data` tiến hành xác thực tính toàn vẹn của schema, 8 đặc trưng số, 1 đặc trưng phân loại (`ocean_proximity`) và biến mục tiêu `median_house_value`. Giai đoạn này chỉ đọc thống kê tổng quan (read-only), tuyệt đối không tự ý điền khuyết hay chuẩn hóa.

In [ ]:
data, target = load_housing_data(ROOT / 'data' / 'raw' / 'housing.csv')
raw_profile = fetch_dataset_info(data, target)

print(f"Tổng số dòng: {raw_profile['shape'][0]:,}, Tổng số cột: {raw_profile['shape'][1]}")
print(f"Số dòng trùng lặp hoàn toàn: {raw_profile['duplicate_rows']}")
display(data.head(5))

raw_info = pd.DataFrame({
    'Kiểu dữ liệu': data.dtypes.astype(str),
    'Số giá trị khuyết': data.isna().sum(),
    'Tỷ lệ khuyết (%)': (data.isna().mean() * 100).round(2)
})
display(raw_info)

print("Thống kê biến mục tiêu (median_house_value):")
display(data[target].describe().to_frame().T)
print(f"Số mẫu đạt trần giá 500,001 USD: {raw_profile['target_max_count']:,} mẫu ({raw_profile['target_max_count']/len(data):.2%})")


## 2. Phân chia tập dữ liệu độc lập ngay từ đầu (Early Split 8:1:1)

Nhằm tuân thủ nguyên tắc chống rò rỉ dữ liệu (Data Leakage), việc chia tập được thực hiện **trước mọi phép học thống kê**. Phân chia theo tỷ lệ 80% Train, 10% Validation, 10% Test với seed cố định. Sử dụng phân tầng (Stratified) theo 5 phân vị của biến mục tiêu để đảm bảo phân phối giá nhà đồng đều giữa các tập.

In [ ]:
splits = split_data_8_1_1(data.drop(columns=target), data[target], random_state=SEED)
X_train, X_val, X_test, y_train, y_val, y_test = splits

# Xác nhận tính độc lập và bảo toàn số lượng dòng
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0
assert len(X_train) + len(X_val) + len(X_test) == len(data)
assert all(X.index.equals(y.index) for X, y in zip(splits[:3], splits[3:]))

display(pd.DataFrame({
    'Tập dữ liệu': ['Train (Huấn luyện)', 'Validation (Kiểm định)', 'Test (Kiểm thử)'],
    'Số mẫu': [len(X_train), len(X_val), len(X_test)],
    'Tỷ lệ': [f"{len(X_train)/len(data):.1%}", f"{len(X_val)/len(data):.1%}", f"{len(X_test)/len(data):.1%}"]
}))
print('Đã phân chia dữ liệu thô độc lập; tập Test được niêm phong hoàn toàn.')


## 3. Khám phá chuyên sâu chỉ trên tập huấn luyện (In-Depth EDA on Train)

Mọi phân tích trực quan hóa chi tiết (phân phối, bản đồ địa lý, tương quan Pearson) **chỉ được tiến hành trên tập Train** (16,512 mẫu) nhằm ngăn ngừa thiên lệch lựa chọn đặc trưng (lookahead bias/data snooping).

In [ ]:
train_frame = X_train.assign(**{TARGET: y_train})
print(f"Thống kê mô tả chi tiết tập Train (N = {len(train_frame):,}):")
display(train_frame.describe(include='all').T)


### 3.1. Phân phối tần suất các đặc trưng số
Biểu đồ phân phối giúp đánh giá độ lệch (skewness) của các biến. Các đặc trưng như `total_rooms`, `total_bedrooms`, `population`, `households` có phân phối lệch phải mạnh (đuôi dài), gợi ý việc tạo các đặc trưng tỷ lệ (ratios) hoặc áp dụng biến đổi log.

In [ ]:
fig_dist = plot_feature_distributions(train_frame)
display(fig_dist)


### 3.2. Bản đồ phân bố không gian địa lý California
Kinh độ (`longitude`) và vĩ độ (`latitude`) kết hợp với màu sắc giá nhà và kích thước điểm theo dân số cho thấy: mức giá cao tập trung rõ nét dọc bờ biển Thái Bình Dương và hai cụm đô thị lớn (vịnh San Francisco và Los Angeles).

In [ ]:
fig_geo = plot_geographical_distribution(train_frame, target_col=TARGET)
display(fig_geo)


### 3.3. Ma trận tương quan nhiệt Pearson (Correlation Heatmap)
Hệ số tương quan tuyến tính giữa các biến số với biến mục tiêu `median_house_value`. `median_income` (thu nhập trung vị) có tương quan thuận mạnh nhất (r ≈ 0.69), đóng vai trò đặc trưng dự báo quan trọng nhất.

In [ ]:
fig_corr = plot_correlation_heatmap(train_frame, target_col=TARGET)
display(fig_corr)

corr_series = train_frame.select_dtypes(include='number').corr()[TARGET].sort_values(ascending=False)
display(corr_series.to_frame('Hệ số tương quan Pearson với target'))


### 3.4. Phân tích biến mục tiêu & đặc trưng phân loại vị trí biển
- Phân phối giá nhà bị chặn trên tại mức 500,001 USD.
- Biểu đồ hộp theo `ocean_proximity` cho thấy nhà ở khu vực `INLAND` (nội địa) có mức giá thấp hơn rõ rệt so với các khu vực tiếp giáp biển (`NEAR OCEAN`, `NEAR BAY`, `<1H OCEAN`).

In [ ]:
fig_target = plot_target_distribution(train_frame, target_col=TARGET)
display(fig_target)

fig_cat = plot_categorical_distribution(train_frame, cat_col='ocean_proximity', target_col=TARGET)
display(fig_cat)


## 4. Thiết lập và khớp bộ tiền xử lý (Fit Preprocessing)

Bộ tiền xử lý `ColumnTransformer` chỉ được khớp (`fit`) trên tập Train:
- **Biến số:** Điền khuyết bằng trung vị (`SimpleImputer(strategy='median')`), tùy chọn tạo tỷ số đặc trưng, và chuẩn hóa Z-score (`StandardScaler`).
- **Biến phân loại (`ocean_proximity`):** Điền giá trị thường gặp nhất và mã hóa One-Hot (`handle_unknown='ignore'`).

In [ ]:
APPLY_LOG = False
CREATE_INTERACTIONS = False

preprocessor = fit_preprocessor(X_train, apply_log=APPLY_LOG, create_interactions=CREATE_INTERACTIONS)
train_features = transform_features(preprocessor, X_train)
val_features = transform_features(preprocessor, X_val)

assert train_features.shape == (len(X_train), 13)
assert val_features.shape == (len(X_val), 13)
assert np.isfinite(train_features).all() and np.isfinite(val_features).all()
print(f"Fit thành công bộ tiền xử lý trên tập Train: số chiều vector đặc trưng = {train_features.shape[1]}.")


## 5. Bàn giao và kiểm tra tính nhất quán (Artifacts Manifest)

Lưu trữ dữ liệu thô đã chia và bộ tiền xử lý vào `notebooks/artifacts/`. Tệp manifest kèm mã băm SHA-256 giúp phát hiện nếu dữ liệu hoặc artifact bị thay đổi ngoài ý muốn.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
save_processed_splits(*splits, target_col=TARGET, output_dir=ARTIFACT_DIR)
save_preprocessor(preprocessor, ARTIFACT_DIR / 'preprocessor.joblib')

manifest = {
    'seed': SEED,
    'apply_log': APPLY_LOG,
    'create_interactions': CREATE_INTERACTIONS,
    'raw_sha256': file_hash(ROOT / 'data' / 'raw' / 'housing.csv'),
    'files': {name: file_hash(ARTIFACT_DIR / name) for name in
              ['train.csv', 'val.csv', 'test.csv', 'preprocessor.joblib']}
}
(ARTIFACT_DIR / 'preparation_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

restored = load_preprocessor(ARTIFACT_DIR / 'preprocessor.joblib')
np.testing.assert_allclose(transform_features(restored, X_val), val_features)
print('Đã lưu dữ liệu thô đã chia và bộ tiền xử lý trong notebooks/artifacts; kiểm tra nạp lại đạt chuẩn.')


## Kết luận giai đoạn

1. Dữ liệu thô gồm 20,640 quan sát, chỉ có cột `total_bedrooms` bị khuyết (207 dòng, ~1.00%), không có dòng trùng lặp.
2. Tập Test và Validation được cách ly độc lập bằng phân tầng 80/10/10.
3. Bộ ảnh EDA toàn diện đã xác định rõ các đặc trưng đuôi dài, tương quan mạnh của `median_income` và sự phụ thuộc vị trí địa lý của giá nhà.
4. Toàn bộ tiền xử lý đã fit sẵn sàng và bàn giao sang `02_mlp_model_training.ipynb`.